##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Get started with Genie 4 interactive worlds

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_Genie_4.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

> **Note:** Genie 4 (`models/genie-4`) is a **paid-tier only** model (~$0.10/second, or ~$6/minute; a ~30-second run in this notebook costs **~$3**). Free-tier API keys are not supported, and live streaming sessions currently serve from the United States. Always call `await conn.complete()` (or `await conn.cancel()`) when finished so server-side generation and billing stop immediately. Check [Gemini API pricing](https://ai.google.dev/gemini-api/docs/pricing) and [rate limits](https://ai.google.dev/gemini-api/docs/rate-limits) for current tier concurrency caps.

**Genie 4** (`models/genie-4`) is a real-time interactive world model available through the **Live Interactions API** (`client.aio.interactions.create` and `client.aio.interactions.connect`). Instead of rendering a fixed offline video clip, Genie 4 generates a continuous `16:9` video stream at `24 fps` paired with `48 kHz` stereo audio while responding live to movement keys (`WASD`), camera look keys (arrow keys), and mid-session text prompts.

Because static notebooks on GitHub cannot host a live interactive `<video>` player, this guide builds a **deterministic Python async WebSocket harness** that:

1. Downloads and center-crops a `16:9` public-domain oil painting from Wikimedia Commons.
2. Initializes a live Genie 4 world with physical contact Foley audio cues.
3. Runs a timed schedule of movement (`key_w`), camera pans (`arrow_left`), and mid-session weather steering (`"Make heavy snow fall across the painted village"` and `"A flock of white doves flies past"`).
4. Saves a **4-frame PIL contact sheet** (`T+2s`, `T+7s`, `T+12s`, `T+18s`) and plays the **48 kHz stereo WAV** stream inline.

If you want to walk through a world interactively with your keyboard before scripting, open [Google AI Studio Live](https://aistudio.google.com/live?model=genie-4).

## Setup

### Install the SDK

Install the latest `google-genai` SDK (with `client.aio.interactions.connect` live WebSocket support) along with `Pillow` for image framing and contact-sheet rendering.

In [ ]:
%pip install -U -q "google-genai>=3.3.0" Pillow

### Set up your API key

Store a **paid-tier** Gemini API key in a Colab Secret named `GEMINI_API_KEY`, and configure the client to route requests to `v1beta` (required for `models/genie-4` Live Interactions). If you have not configured Colab Secrets yet, follow the [Authentication quickstart](./Authentication.ipynb).

In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options={"api_version": "v1beta"},
)

if not hasattr(client.aio.interactions, "connect"):
    raise RuntimeError(
        "Your installed google-genai version does not expose "
        "client.aio.interactions.connect yet. Upgrade google-genai to the "
        "Genie 4 release and restart the runtime."
    )

### Select the model

Select `models/genie-4` for interactive world generation:

In [ ]:
MODEL_ID = "models/genie-4"  # @param ["models/genie-4"] {"allow-input": true, "isTemplate": true}

## Step inside a living oil painting over WebSocket

A live Genie 4 session uses three client event types over `client.aio.interactions.connect`:

* **`world_setup`**: Sent once after connecting to initialize the world from a `world_prompt`, a `16:9` `reference_image`, or both.
* **`user_action`**: Real-time first-person movement (`key_w`, `key_a`, `key_s`, `key_d`) and camera look (`arrow_up`, `arrow_down`, `arrow_left`, `arrow_right`) with `action` set to `"key_down"`, `"key_up"`, or `"key_click"`.
* **`input`**: Mid-session natural-language prompts that steer weather, lighting, objects, or characters while you move.

> **Important:** Genie 4 does not accept audio input, video input, tool declarations, or `system_instruction` fields. Keep all initial scene direction inside `world_setup`.

### Prepare a 16:9 public-domain reference painting

The cell below downloads Vincent van Gogh's [*Starry Night Over the Rhône* (1888)](https://commons.wikimedia.org/wiki/File:Starry_Night_Over_the_Rhone.jpg), a **public-domain** oil painting hosted on **Wikimedia Commons**. It center-crops the canvas to `16:9` (`640x360`) and compresses the JPEG so the base64 payload stays under `48 KB` (compact over WebSocket and small enough to reuse inside a `64 KB` browser WebRTC DataChannel message).

Stick to `16:9` public-domain landscape artwork when seeding worlds: portrait images crop or invent side walls, and street photographs containing real bystanders, brand logos, or copyrighted game IP can trigger mid-session safety termination.

In [ ]:
import base64
import io
import urllib.request
from IPython.display import display
from PIL import Image

PAINTING_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/9/94/"
    "Starry_Night_Over_the_Rhone.jpg/960px-Starry_Night_Over_the_Rhone.jpg"
)


def load_16x9_seed_painting(
    url: str,
    size: tuple[int, int] = (640, 360),
    max_b64_bytes: int = 48_000,
) -> tuple[Image.Image, str]:
    """Downloads an image, center-crops to 16:9, and encodes JPEG under max_b64_bytes."""
    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": (
                "GeminiCookbook/1.0 (https://github.com/google-gemini/cookbook)"
            )
        },
    )
    with urllib.request.urlopen(request, timeout=20) as response:
        source_img = Image.open(io.BytesIO(response.read())).convert("RGB")

    width, height = source_img.size
    target_height = int(width * 9 / 16)
    if target_height <= height:
        top = (height - target_height) // 2
        cropped = source_img.crop((0, top, width, top + target_height))
    else:
        target_width = int(height * 16 / 9)
        left = (width - target_width) // 2
        cropped = source_img.crop((left, 0, left + target_width, height))

    resized = cropped.resize(size, Image.Resampling.LANCZOS)
    encoded_b64 = ""
    for quality in (65, 55, 45, 38, 30, 25, 20, 15):
        buffer = io.BytesIO()
        resized.save(buffer, format="JPEG", quality=quality, optimize=True)
        encoded_b64 = base64.b64encode(buffer.getvalue()).decode("ascii")
        if len(encoded_b64) <= max_b64_bytes:
            break
    else:
        raise ValueError(
            f"Compressed image ({len(encoded_b64):,} bytes) exceeds {max_b64_bytes:,}-byte budget."
        )

    return resized, encoded_b64


seed_painting, reference_b64 = load_16x9_seed_painting(PAINTING_URL)
print(
    f"Prepared 16:9 seed image ({seed_painting.size[0]}x{seed_painting.size[1]}), "
    f"base64 payload: {len(reference_b64):,} bytes"
)
display(seed_painting)

### Define the physical Foley prompt and beat schedule

When writing `world_prompt`, follow two prompting practices tailored to Genie 4's planner:

1. **Give forward motion a visible physical cause at frame 0** (standing on a cobblestone canal bridge in first-person walking POV) and write **broadband contact Foley** (`water lapping`, `leaves rustling`, `footsteps on cobblestone`). Avoid narrowband or musical terms (`music`, `synth`, `hum`, `rumble`, `drone`, `roar`), which the audio planner rewrites into physical contact sounds.
2. **Repeat mid-session text steering across step boundaries**: Genie 4 samples text steering once per generation step, so sending each `input` event `10` times spaced `150 ms` apart (~1.5 seconds total) ensures reliable uptake without hard-cut prefixes.

In [ ]:
WORLD_PROMPT = """
    Immersive first-person POV standing on a cobblestone canal bridge inside a
    vibrant, thick-impasto post-impressionist oil painting of a starry riverside
    village at twilight. Swirling cobalt-blue sky currents and glowing golden
    cafe lanterns shimmer with living, tactile brushstroke textures that ripple
    gently in the breeze, with a wooden rowboat moored at the stone steps ahead.
    Audio: Gentle water lapping against the stone steps, leaves rustling in a
    soft evening breeze, footsteps on cobblestone.
"""

SCRIPTED_BEATS = [
    {"t": 0.0, "kind": "user_action", "action": "key_down", "input": "key_w"},
    {
        "t": 4.0,
        "kind": "user_action",
        "action": "key_click",
        "input": "arrow_left",
        "count": 3,
    },
    {
        "t": 8.0,
        "kind": "text",
        "text": "Make heavy snow fall across the painted village",
    },
    {
        "t": 16.0,
        "kind": "text",
        "text": "A flock of white doves flies past",
    },
    {"t": 24.0, "kind": "user_action", "action": "key_up", "input": "key_w"},
    {"t": 28.0, "kind": "complete"},
]

### Stream frames and audio with the Live Interactions API

Run the live WebSocket session below (~28 seconds after the first frame arrives, **~$3** on a paid-tier API key).

Notice two lifecycle details in `run_living_painting_session`:

* **Beat timing waits for the first video frame**: Initial world planning can take up to 15 seconds before frame 1 arrives, so `first_frame_ready` anchors the beat schedule to actual media playback rather than socket connection time.
* **Explicit `await conn.complete()` stops billing**: Exiting `async with client.aio.interactions.connect(...)` detaches the local WebSocket without ending the server-side session. Calling `await conn.complete()` (and falling back to `await conn.cancel()` on interruption) ensures server generation and billing stop immediately.

In [ ]:
import asyncio


async def send_steered_prompt(
    conn,
    prompt_text: str,
    repeats: int = 10,
    interval_sec: float = 0.15,
) -> None:
    """Repeats a mid-session text input across step boundaries for reliable uptake."""
    event = {
        "event_type": "input",
        "content": [{"type": "text", "text": prompt_text}],
    }
    for idx in range(repeats):
        await conn.send(event)
        if idx + 1 < repeats:
            await asyncio.sleep(interval_sec)


async def run_beat_schedule(
    conn,
    beats: list[dict],
    first_frame_ready: asyncio.Event,
) -> list[tuple[float, str]]:
    """Runs deterministic movement and steering beats anchored to frame 1."""
    try:
        await first_frame_ready.wait()
        loop = asyncio.get_running_loop()
        start_time = loop.time()
        executed_log: list[tuple[float, str]] = []

        for beat in beats:
            delay = (start_time + beat["t"]) - loop.time()
            if delay > 0:
                await asyncio.sleep(delay)

            elapsed = loop.time() - start_time
            kind = beat["kind"]
            if kind == "user_action":
                repeats = beat.get("count", 1)
                for click_idx in range(repeats):
                    await conn.send(
                        {
                            "event_type": "user_action",
                            "action": beat["action"],
                            "input": beat["input"],
                        }
                    )
                    if click_idx + 1 < repeats:
                        await asyncio.sleep(0.12)
                executed_log.append(
                    (elapsed, f"{beat['action']} {beat['input']} (x{repeats})")
                )
            elif kind == "text":
                executed_log.append((elapsed, f"input: {beat['text']}"))
                await send_steered_prompt(conn, beat["text"])
            elif kind == "complete":
                await conn.complete()
                executed_log.append((elapsed, "interaction.complete"))
                break

        return executed_log
    except asyncio.CancelledError:
        raise
    except Exception:
        try:
            await conn.cancel()
        finally:
            await conn.detach()
        raise


async def run_living_painting_session(
    model_id: str,
    world_prompt: str,
    beats: list[dict],
    reference_jpeg_b64: str | None = None,
    max_session_sec: float = 75.0,
) -> tuple[list[tuple[float, bytes]], list[bytes], int, int]:
    """Creates a Genie 4 interaction and captures JPEG frames and stereo PCM."""
    interaction = await client.aio.interactions.create(
        model=model_id,
        api_version="v1beta",
        timeout=30.0,
    )
    print(f"Created interaction: {interaction.id}")

    timed_frames: list[tuple[float, bytes]] = []
    audio_chunks: list[bytes] = []
    sample_rate = 48_000
    channels = 2
    first_frame_ready = asyncio.Event()
    completed = False

    try:
        async with client.aio.interactions.connect(
            interaction.id,
            api_version="v1beta",
            timeout=30.0,
        ) as conn:
            beat_task = asyncio.create_task(
                run_beat_schedule(conn, beats, first_frame_ready)
            )
            try:
                async with asyncio.timeout(max_session_sec):
                    setup_event: dict[str, object] = {
                        "event_type": "world_setup",
                        "world_prompt": {
                            "type": "text",
                            "text": " ".join(world_prompt.split()),
                        },
                    }
                    if reference_jpeg_b64:
                        setup_event["reference_image"] = {
                            "type": "image",
                            "mime_type": "image/jpeg",
                            "data": reference_jpeg_b64,
                        }
                    await conn.send(setup_event)

                    async for event in conn.receive():
                        etype = event.event_type
                        if etype == "step.delta":
                            delta = event.delta
                            dtype = getattr(delta, "type", None)
                            data_b64 = getattr(delta, "data", None)
                            if dtype == "video" and isinstance(data_b64, str):
                                if not first_frame_ready.is_set():
                                    first_frame_ready.set()
                                media_timestamp = len(timed_frames) / 24.0
                                timed_frames.append(
                                    (media_timestamp, base64.b64decode(data_b64))
                                )
                            elif dtype == "audio" and isinstance(data_b64, str):
                                sr = getattr(delta, "sample_rate", None) or getattr(
                                    delta, "rate", None
                                )
                                ch = getattr(delta, "channels", None)
                                if isinstance(sr, int) and sr > 0:
                                    sample_rate = sr
                                if isinstance(ch, int) and ch > 0:
                                    channels = ch
                                audio_chunks.append(base64.b64decode(data_b64))
                        elif etype == "error":
                            raise RuntimeError(f"Live interaction error: {event}")
                        elif etype in (
                            "interaction.completed",
                            "interaction.canceled",
                            "interaction.cancelled",
                        ):
                            completed = True
                            break

                    await asyncio.sleep(0)
                    if beat_task.done():
                        beat_exc = (
                            None if beat_task.cancelled() else beat_task.exception()
                        )
                        if (
                            completed
                            and isinstance(beat_exc, RuntimeError)
                            and (
                                "terminal" in str(beat_exc)
                                or "detached" in str(beat_exc)
                            )
                        ):
                            print(
                                "Warning: Server ended the session before all "
                                "scripted beats completed."
                            )
                        else:
                            for beat_t, description in await beat_task:
                                print(f"  [T+{beat_t:4.1f}s] {description}")
                    elif completed:
                        print(
                            "Warning: Server ended the session before all scripted "
                            "beats completed."
                        )
            finally:
                if not beat_task.done():
                    beat_task.cancel()
                await asyncio.gather(beat_task, return_exceptions=True)
                if not completed:
                    try:
                        await conn.cancel()
                    except Exception:
                        pass
                    else:
                        completed = True
    finally:
        if not completed:
            try:
                await client.aio.interactions.cancel(
                    interaction.id,
                    api_version="v1beta",
                    timeout=15.0,
                )
            except Exception:
                pass

    duration_sec = len(timed_frames) / 24.0
    print(
        f"Captured {len(timed_frames)} JPEG frames ({duration_sec:.1f}s @ 24 fps) "
        f"and {len(audio_chunks)} PCM audio chunks ({sample_rate} Hz, {channels} ch)."
    )
    return timed_frames, audio_chunks, sample_rate, channels


timed_frames, audio_chunks, sample_rate, channels = (
    await run_living_painting_session(
        model_id=MODEL_ID,
        world_prompt=WORLD_PROMPT,
        beats=SCRIPTED_BEATS,
        reference_jpeg_b64=reference_b64,
    )
)

### Render the 4-frame contact sheet and stereo audio

Assemble a `2x2` contact sheet sampling frames at `T+2s`, `T+7s`, `T+12s`, and `T+18s` to inspect the stream across each scheduled beat, then play back the captured 16-bit PCM WAV audio.

In [ ]:
import wave
from IPython.display import Audio
from PIL import ImageDraw

SNAPSHOT_TARGETS = [
    (2.0, "T+2s | Beat 1: forward walk (key_w)"),
    (7.0, "T+7s | Beat 2: camera pan left"),
    (12.0, "T+12s | Beat 3: snow prompt sent"),
    (18.0, "T+18s | Beat 4: doves prompt sent"),
]


def build_contact_sheet(
    frames: list[tuple[float, bytes]],
    targets: list[tuple[float, str]],
    tile_size: tuple[int, int] = (640, 360),
) -> Image.Image:
    """Renders a 2x2 contact sheet from the closest captured JPEG frames."""
    if not frames:
        raise RuntimeError("No video frames captured to build a contact sheet.")

    tile_w, tile_h = tile_size
    sheet = Image.new("RGB", (tile_w * 2, tile_h * 2), color=(15, 17, 21))
    draw = ImageDraw.Draw(sheet)

    for idx, (target_sec, label) in enumerate(targets):
        actual_sec, raw_jpeg = min(
            frames, key=lambda item: abs(item[0] - target_sec)
        )
        tile = (
            Image.open(io.BytesIO(raw_jpeg))
            .convert("RGB")
            .resize(tile_size, Image.Resampling.LANCZOS)
        )
        col, row = idx % 2, idx // 2
        x0, y0 = col * tile_w, row * tile_h
        sheet.paste(tile, (x0, y0))

        caption = f"{label} (frame T+{actual_sec:.1f}s)"
        draw.rectangle((x0 + 12, y0 + 12, x0 + 460, y0 + 42), fill=(0, 0, 0))
        draw.text((x0 + 20, y0 + 20), caption, fill=(255, 255, 255))

    return sheet


contact_sheet = build_contact_sheet(timed_frames, SNAPSHOT_TARGETS)
contact_sheet.save("/tmp/genie4_contact_sheet.jpg", format="JPEG", quality=90)
display(contact_sheet)

if audio_chunks:
    WAV_OUTPUT_PATH = "/tmp/genie4_world_audio.wav"
    with wave.open(WAV_OUTPUT_PATH, "wb") as wav_out:
        wav_out.setnchannels(channels)
        wav_out.setsampwidth(2)
        wav_out.setframerate(sample_rate)
        wav_out.writeframes(b"".join(audio_chunks))
    display(Audio(WAV_OUTPUT_PATH, rate=sample_rate))
else:
    print("No audio chunks arrived before the session ended.")

## Scout interactively in AI Studio, then script

Trying to tune camera angles or prompt timing purely from batch code runs costs ~$3 per 30-second pass. A faster workflow splits exploration from automation:

1. **Scout interactively first**: Open [Google AI Studio Live](https://aistudio.google.com/live?model=genie-4), paste your `world_prompt` (and optional `16:9` painting seed), and test `WASD` movement, arrow-key camera turns, and live text prompts interactively.
2. **Capture reproducible-ish evaluation passes in Python**: Once you confirm which prompts and Foley cues behave well, encode them as a `SCRIPTED_BEATS` list in this WebSocket harness to run automated agent loops, side-by-side prompt regression checks, or latency measurements.

Keep in mind that Genie 4 world simulation is **stochastic**: running the exact same `SCRIPTED_BEATS` schedule twice follows the same action timing, while producing a fresh visual and audio take each time.

## Author causal world rules ("Chekhov's lever")

In addition to direct mid-session weather edits, you can declare **causal rules** inside `world_prompt` at step 0 (`Rule: ...`) that stay dormant until triggered by a mid-session `input` event:

```text
A spaceship bridge with a big red lever on a console far away from me. Rule: Pulling the red lever makes the stars outside the window stretch into bright hyperspace streaks.
```

When authoring causal rules, apply three phrasing patterns verified in testing:

* **Anchor triggers to an object or character, never a location**: Object and character rules fire reliably when prompted (for example, sending `"Pull the red lever"` at `T+15s`), whereas location-based triggers (`"When entering the hallway..."`) frequently fail.
* **Place trigger objects `"far away from me"`**: Objects spawned right next to the camera can self-trigger prematurely during ambient motion unless distanced in the initial prompt.
* **Add `"waits until a rule is triggered"` for characters**: When staging an interactive character, explicitly state that the character waits so the planner holds the action until your `input` prompt arrives.

Treat these phrasings as practical prompting workarounds rather than deterministic game-engine state machines. You can pass `LEVER_WORLD_PROMPT` and `LEVER_BEATS` directly into `run_living_painting_session(MODEL_ID, LEVER_WORLD_PROMPT, LEVER_BEATS)` (omitting `reference_jpeg_b64` for a text-only world):

In [ ]:
LEVER_WORLD_PROMPT = (
    "A spaceship bridge with a big red lever on a console far away from me. "
    "Rule: Pulling the red lever makes the stars outside the window stretch "
    "into bright hyperspace streaks."
)

LEVER_BEATS = [
    {"t": 0.0, "kind": "user_action", "action": "key_down", "input": "key_w"},
    {"t": 6.0, "kind": "user_action", "action": "key_up", "input": "key_w"},
    {"t": 15.0, "kind": "text", "text": "Pull the red lever"},
    {"t": 25.0, "kind": "complete"},
]
print("Configured causal rule prompt:", LEVER_WORLD_PROMPT)

## Build browser apps with ephemeral auth tokens

Python's `google-genai` SDK connects over **WebSocket**, which is ideal for server-side automation, evaluation harnesses, and notebooks. When building an interactive web application where end users steer live in their browser, **never expose `GEMINI_API_KEY` in client-side code**.

Instead, use a two-tier flow:

1. **Backend server (Python or Node.js)**: Creates the interaction with `client.interactions.create(model=MODEL_ID, api_version="v1beta")` (`serverAi.interactions.create({ model: 'models/genie-4', api_version: 'v1beta' })` in Node.js) and mints a short-lived **ephemeral auth token** (`auth_tokens/...`) scoped to that interaction via `client.interactions.auth_tokens.create` in Python or `serverAi.interactions.authTokens.create` in Node.js.
2. **Browser client (`@google/genai` JavaScript/TypeScript SDK)**: Initializes `new GoogleGenAI({ apiKey: authToken, apiVersion: 'v1beta' })` and connects over **WebSocket** (`{ api_version: 'v1beta' }`, which works across restrictive corporate networks) or **WebRTC** (`{ transport: 'webrtc', api_version: 'v1beta' }`, which decodes RTP video and Opus stereo audio directly into a `<video>` element).

Here is the Python backend helper that mints browser session credentials:

In [ ]:
def mint_ephemeral_browser_session(model_id: str = MODEL_ID) -> dict[str, str]:
    """Creates a Genie 4 interaction and returns a scoped ephemeral auth token."""
    session = client.interactions.create(
        model=model_id,
        api_version="v1beta",
        timeout=30.0,
    )
    token_resp = client.interactions.auth_tokens.create(
        id=session.id,
        api_version="v1beta",
        timeout=30.0,
    )
    return {
        "interaction_id": session.id,
        "auth_token": token_resp.auth_token or "",
    }

Your browser frontend then exchanges `{ interaction_id, auth_token }` for a live stream using `@google/genai`—keeping `session.complete()` inside an explicit Stop button or `pagehide` handler and tearing down DOM listeners cleanly when the stream ends:

```typescript
import { GoogleGenAI } from '@google/genai';

// Initialize in the browser using the backend-minted ephemeral auth_token:
const clientAi = new GoogleGenAI({ apiKey: authToken, apiVersion: 'v1beta' });
const frameImg = document.querySelector<HTMLImageElement>('#world-frame')!;
const stopBtn = document.querySelector<HTMLButtonElement>('#stop-btn')!;

// Connect over browser WebSocket (universal HTTPS/WSS fallback):
const session = await clientAi.interactions.connect(
  interactionId,
  { api_version: 'v1beta' },
  { timeout_ms: 30_000 },
);
// Or, where WebRTC UDP media is enabled, attach an HTMLVideoElement directly:
// const session = await clientAi.interactions.connect(
//   interactionId,
//   { transport: 'webrtc', api_version: 'v1beta' },
//   { timeout_ms: 30_000 },
// );
// session.transport.setVideoElement(document.querySelector('#world-video')!);

await session.send({
  event_type: 'world_setup',
  world_prompt: {
    type: 'text',
    text: 'Immersive first-person POV standing on a cobblestone canal bridge inside a living oil painting.',
  },
});

const cleanup = new AbortController();
let wHeld = false;

window.addEventListener(
  'keydown',
  (e) => {
    if (e.repeat || e.code !== 'KeyW' || wHeld) return;
    wHeld = true;
    void session
      .send({ event_type: 'user_action', action: 'key_down', input: 'key_w' })
      .catch(console.error);
  },
  { signal: cleanup.signal },
);

window.addEventListener(
  'keyup',
  (e) => {
    if (e.code !== 'KeyW' || !wHeld) return;
    wHeld = false;
    void session
      .send({ event_type: 'user_action', action: 'key_up', input: 'key_w' })
      .catch(console.error);
  },
  { signal: cleanup.signal },
);

// End the server-side interaction and stop billing when the user clicks Stop:
stopBtn.addEventListener(
  'click',
  () => {
    void session.complete().catch(console.error);
  },
  { signal: cleanup.signal },
);

try {
  for await (const event of session.receive()) {
    if (event.event_type === 'step.delta' && event.delta?.type === 'video') {
      frameImg.src = `data:image/jpeg;base64,${event.delta.data}`;
    } else if (
      event.event_type === 'interaction.completed' ||
      event.event_type === 'error'
    ) {
      break;
    }
  }
} finally {
  cleanup.abort();
}
```

## Understand Genie 4 limitations

Genie 4 generates a controllable world, with four limits you will hit in your first minute:

* **It forgets what is behind you**: Turn 180° and the scene is regenerated rather than recalled, so treat each run as a forward-moving stage rather than a persistent 3D map.
* **It is never still**: The planner injects ambient micro-kinetics—dust, steam, spray, and foliage motion—every step by design, which reads as continuous scene activity even when you release movement keys.
* **It starts slow and stops hard**: The first frame can take up to 15 seconds while the world initializes, and the session ends at the session cap with no resumption.
* **It rewrites your audio prompt**: Narrowband or musical words (`hum`, `rumble`, `synth`, unrequested `music`) are replaced with physical contact sounds—write Foley, not soundtrack.

Within those limits, one mid-session prompt changes the weather over a village you are walking through, and no offline video model can do that.

## What's next

* Try interactive keyboard world exploration directly in [Google AI Studio Live](https://aistudio.google.com/live?model=genie-4).
* Explore conversational offline video generation and multi-turn editing in the [Gemini Omni Flash quickstart](./Get_started_Omni.ipynb) and studio clip synthesis in the [Veo quickstart](./Get_started_Veo.ipynb).
* Build low-latency bidirectional voice and camera assistants with the [Gemini Live API quickstart](./Get_started_LiveAPI.ipynb).
* Review the [Interactions API overview](./Get_started_interactions_api.ipynb), [Interactions API documentation](https://ai.google.dev/gemini-api/docs/interactions), and [Gemini API pricing](https://ai.google.dev/gemini-api/docs/pricing).